## 1. Find post sitemaps

In [1]:
# Find Kontrast post sitemaps in the sitemap index.
import re
import urllib.request
import xml.etree.ElementTree as ET
from urllib.parse import urlparse

index_url = "https://kontrast.at/sitemap_index.xml"
request = urllib.request.Request(index_url, headers={"User-Agent": "news-analytics-scraper/0.1"})
with urllib.request.urlopen(request, timeout=20) as response:
    root = ET.fromstring(response.read())

post_sitemaps = []
for loc in root.findall(".//{*}sitemap/{*}loc"):
    sitemap_url = loc.text.strip()
    filename = urlparse(sitemap_url).path.rsplit("/", 1)[-1]
    if re.fullmatch(r"post-sitemap(?:\d+)?\.xml", filename):
        post_sitemaps.append(sitemap_url)
        print(sitemap_url)

https://kontrast.at/post-sitemap.xml
https://kontrast.at/post-sitemap2.xml
https://kontrast.at/post-sitemap3.xml
https://kontrast.at/post-sitemap4.xml


## 2. Inspect article URLs and lastmod

In [2]:
# Preview article URLs and lastmod values from the first post sitemap.
request = urllib.request.Request(post_sitemaps[0], headers={"User-Agent": "news-analytics-scraper/0.1"})
with urllib.request.urlopen(request, timeout=20) as response:
    post_root = ET.fromstring(response.read())

print(f"Articles from {post_sitemaps[0]}:")
for article in post_root.findall("{*}url")[:5]:
    print(f"{article.findtext('{*}loc')} | lastmod={article.findtext('{*}lastmod')}")

Articles from https://kontrast.at/post-sitemap.xml:
https://kontrast.at/wie-studieren-schlau-und-nicht-arm-macht/ | lastmod=2017-04-12T21:00:53+00:00
https://kontrast.at/streit-um-die-maut-wie-die-fpoe-in-bruessel-oesterreichs-autofahrer-verraet/ | lastmod=2017-04-12T21:00:53+00:00
https://kontrast.at/die-bemuehungen-nicht-kleinreden-dominik-bernhofer-und-sven-hergovich-antworten-auf-niki-kowall/ | lastmod=2017-04-12T21:00:53+00:00
https://kontrast.at/warum-aus-aleppo-mehr-propaganda-als-nachrichten-zu-uns-kam/ | lastmod=2017-04-12T21:43:13+00:00
https://kontrast.at/sorgen-frauen-fuer-weltfrieden/ | lastmod=2017-04-12T21:53:07+00:00


## 3. Fetch one article as raw HTML

In [13]:
# Fetch the first article as raw HTML bytes.
article_url = post_root.findtext("{*}url/{*}loc")
request = urllib.request.Request(article_url, headers={"User-Agent": "news-analytics-scraper/0.1"})
with urllib.request.urlopen(request, timeout=20) as response:
    html = response.read()
    print(f"Fetched {response.url}: HTTP {response.status}, {len(html)} bytes")
    print(f"Content-Type: {response.headers.get('Content-Type')}")
    # Save the downloaded HTML using the URL's final path segment.
    import os
    os.makedirs("debug_articles", exist_ok=True)
    slug = urlparse(article_url).path.rstrip("/").rsplit("/", 1)[-1]
    filename = os.path.join("debug_articles", slug + ".html")
    with open(filename, "wb") as f:
        f.write(html)
    print(os.path.abspath(filename))

Fetched https://kontrast.at/wie-studieren-schlau-und-nicht-arm-macht/: HTTP 200, 484636 bytes
Content-Type: text/html; charset=UTF-8

debug_articles/.html
https://kontrast.at/wie-studieren-schlau-und-nicht-arm-macht/
